# Phase 6.3.2 — Evidence Applicability Audit & Guarded Decision
Zero new LLM calls. Reuse the exact Phase 5.2, 6.3B and 6.3.1 archives. No manual uploads, API keys, embedding models, new retrieval or library changes. Lexical flags are triage hints, not semantic labels. Guarded decision is a deliberately conservative offline ablation, not a final classifier.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'simplified-rag-v3'
ROOT = Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC = ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent:',tfl_agent.__file__)
subprocess.run([sys.executable,'-m','pytest','-q',
                str(ROOT/'tests/test_phase6_evidence_applicability.py'),
                str(ROOT/'tests/test_phase6_decision_replay.py'),
                str(ROOT/'tests/test_phase6_llm_pilot.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_decision_replay import ARCHIVE_NAME, ARCHIVE_SHA, _sha
from tfl_agent.agent.phase6_evidence_applicability import (
    REPLAY_NAME, REPLAY_SHA, HISTORICAL_SHA, run_applicability)
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
def frozen_file(folder, name, sha):
    # Only select a file with the frozen hash. Supports Drive duplicate-name suffixes.
    p=BASE/folder/name
    if p.is_file() and _sha(p)==sha:
        return p
    matches=[x for x in p.parent.glob('*.zip') if _sha(x)==sha]
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected {p} with SHA256 {sha}; matching archives: {matches}')
    return matches[0]
PILOT_ZIP=frozen_file('Phase63_LLM_Pilot_Outputs',ARCHIVE_NAME,ARCHIVE_SHA)
REPLAY_ZIP=frozen_file('Phase631_Decision_Replay_Outputs',REPLAY_NAME,REPLAY_SHA)
HISTORICAL_ZIP=frozen_file('Phase52_Outputs','20261127T021948099654Z.zip',HISTORICAL_SHA)
for p in (PILOT_ZIP,REPLAY_ZIP,HISTORICAL_ZIP):
    print('Verified:',p)
print('Authorized new LLM calls: 0')

In [ ]:
import pandas as pd
from IPython.display import display
CACHE=Path('/content/Phase632_Cache')/RUN_ID
OUT=Path('/content/Phase632_Evidence_Applicability')/RUN_ID
report,pairs=run_applicability(PILOT_ZIP,REPLAY_ZIP,HISTORICAL_ZIP,CACHE,OUT)
assert report['new_llm_calls']==0
assert report['pilot_responses']==18 and report['historical_responses']==44
assert len(pairs)==12 and not any(r['generation_authorized'] for r in pairs)
print(json.dumps({k:v for k,v in report.items() if k!='metrics'},ensure_ascii=False,indent=2))
metrics=[]
for arm, m in report['metrics'].items():
    for method, stats in m.items():
        if method!='gate_actions':
            metrics.append({'arm':arm,'method':method,**stats})
display(pd.DataFrame(metrics))
display(pd.DataFrame(pairs)[['study_id','entry_id','arm','expected','title_recommendation',
                            'sap_recommendation','sap_supported_template','guarded_recommendation',
                            'guarded_supported_template','decision_policy']])
print('Independent semantic labels:',report['independent_applicability_labels'])
print('Guarded results are policy ablations, not evidence-validated predictions.')

In [ ]:
DEST=BASE/'Phase632_Evidence_Applicability_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(p.name for p in OUT.iterdir()))
print('Share the ZIP only if authorized to disclose the SAP excerpts and historical responses.')

## Interpretation
- Pilot_Citation_Applicability.csv: citation occurrence, original candidate assessment, local surrounding text, exact match and triage flags.
- Historical_Claim_Applicability.csv and Historical_Response_Inventory.csv: all 44 saved study/target initial and completion responses, including rejected claims and prior status matches. Claim occurrences are not independent samples.
- Applicability_Review_Queue.csv: unlabeled evidence occurrences for offline evaluation, not required runtime human approval. The hints do not constitute review labels.
- Guarded_Decisions.csv: 12 matched comparisons, no gold used by the guard. Recommendation defaults to title and support requires agreement without uncertainty. A useful SAP override cannot yet pass this conservative policy.
- No preset, specification or DOCX authorization. Do not start a new LLM experiment without explicit permission.